# Rolling Feature Data Leakage

In [32]:
# Imports
import numpy as np
import pandas as pd
import os

## Load a sample from energy dataset

In [33]:
# Load the energy dataset (first 100 rows)
current_dir = os.getcwd()
files_dir = current_dir + '/data/'

power_pd = pd.read_csv(files_dir + 'energy_dataset.csv', header=0, nrows=100)

# Select only the 'total load actual' and 'price actual' columns for demo
df = power_pd[['time', 'total load actual', 'price actual']].copy()
df.columns = ['time', 'total_load_actual', 'price_actual']
df['time'] = pd.to_datetime(df['time'])
df = df.reset_index(drop=True)

print(f"Dataset shape: {df.shape}")
df.head(10)

Dataset shape: (100, 3)


,time,total_load_actual,price_actual
0,2015-01-01 00:00:00+01:00,25385.0,65.41
1,2015-01-01 01:00:00+01:00,24382.0,64.92
2,2015-01-01 02:00:00+01:00,22734.0,64.48
3,2015-01-01 03:00:00+01:00,21286.0,59.32
4,2015-01-01 04:00:00+01:00,20264.0,56.04
5,2015-01-01 05:00:00+01:00,19905.0,53.63
6,2015-01-01 06:00:00+01:00,20010.0,51.73
7,2015-01-01 07:00:00+01:00,20377.0,51.43
8,2015-01-01 08:00:00+01:00,20094.0,48.98
9,2015-01-01 09:00:00+01:00,20637.0,54.20


---
## What the Notebook Currently Does

Computes rolling averages on all data first, then splits into train/test.

In [34]:
# Bug: Compute rolling on ALL data first
df_wrong = df.copy()

# This is what the notebook does:
df_wrong['price_rolling3h'] = df_wrong['price_actual'].rolling(window=3, min_periods=1).mean()

print("Rolling computed on entire dataset:")
df_wrong[['time', 'price_actual', 'price_rolling3h']].head(10)

Rolling computed on entire dataset:


,time,price_actual,price_rolling3h
0,2015-01-01 00:00:00+01:00,65.41,65.410000
1,2015-01-01 01:00:00+01:00,64.92,65.165000
2,2015-01-01 02:00:00+01:00,64.48,64.936667
3,2015-01-01 03:00:00+01:00,59.32,62.906667
4,2015-01-01 04:00:00+01:00,56.04,59.946667
5,2015-01-01 05:00:00+01:00,53.63,56.330000
6,2015-01-01 06:00:00+01:00,51.73,53.800000
7,2015-01-01 07:00:00+01:00,51.43,52.263333
8,2015-01-01 08:00:00+01:00,48.98,50.713333
9,2015-01-01 09:00:00+01:00,54.20,51.536667


In [35]:
# Split into train/test at row 70 (70/30 split)
split_idx = 70

print(f"\nRows {split_idx-2} to {split_idx+2}:")
print(df_wrong.loc[split_idx-2:split_idx+2, ['time', 'price_actual', 'price_rolling3h']])
print(f"\nRow 0-{split_idx-1} = TRAIN | Row {split_idx}-99 = TEST")


Rows 68 to 72:
                        time  price_actual  price_rolling3h
68 2015-01-03 20:00:00+01:00         72.45        72.030000
69 2015-01-03 21:00:00+01:00         72.86        72.483333
70 2015-01-03 22:00:00+01:00         70.10        71.803333
71 2015-01-03 23:00:00+01:00         67.62        70.193333
72 2015-01-04 00:00:00+01:00         55.22        64.313333

Row 0-69 = TRAIN | Row 70-99 = TEST


In [36]:
# Show the DATA LEAKAGE for first TEST row
print(f"\nFirst TEST row (row {split_idx}):")
print(f"  price_actual (TARGET we want to predict) = {df_wrong.loc[split_idx, 'price_actual']:.2f}")
print(f"  price_rolling3h (FEATURE given to model)  = {df_wrong.loc[split_idx, 'price_rolling3h']:.2f}")

print(f"\nHow was price_rolling3h calculated?")
print(f"  Row {split_idx-2}: price = {df_wrong.loc[split_idx-2, 'price_actual']:.2f}")
print(f"  Row {split_idx-1}: price = {df_wrong.loc[split_idx-1, 'price_actual']:.2f}")
print(f"  Row {split_idx}: price = {df_wrong.loc[split_idx, 'price_actual']:.2f}  (Target)")

manual_calc = df_wrong.loc[split_idx-2:split_idx, 'price_actual'].mean()
print(f"\n  Rolling mean = ({df_wrong.loc[split_idx-2, 'price_actual']:.2f} + {df_wrong.loc[split_idx-1, 'price_actual']:.2f} + {df_wrong.loc[split_idx, 'price_actual']:.2f}) / 3 = {manual_calc:.2f}")

print(f"\nThe model is being asked to predict {df_wrong.loc[split_idx, 'price_actual']:.2f}")
print(f"But it's given a feature ({df_wrong.loc[split_idx, 'price_rolling3h']:.2f}) that includes the target")


First TEST row (row 70):
  price_actual (TARGET we want to predict) = 70.10
  price_rolling3h (FEATURE given to model)  = 71.80

How was price_rolling3h calculated?
  Row 68: price = 72.45
  Row 69: price = 72.86
  Row 70: price = 70.10  (Target)

  Rolling mean = (72.45 + 72.86 + 70.10) / 3 = 71.80

The model is being asked to predict 70.10
But it's given a feature (71.80) that includes the target


---
## Fix: Use only past data for rolling

Add `.shift(1)` before rolling to ensure we only use values from before the current row.

In [37]:
df_correct = df.copy()

# shift(1) moves all values down by 1 row, so rolling only sees PAST values
df_correct['price_rolling3h'] = df_correct['price_actual'].shift(1).rolling(window=3, min_periods=1).mean()

print("Rolling computed with shift(1) - only uses past data:")
df_correct[['time', 'price_actual', 'price_rolling3h']].head(10)

Rolling computed with shift(1) - only uses past data:


,time,price_actual,price_rolling3h
0,2015-01-01 00:00:00+01:00,65.41,NaN
1,2015-01-01 01:00:00+01:00,64.92,65.410000
2,2015-01-01 02:00:00+01:00,64.48,65.165000
3,2015-01-01 03:00:00+01:00,59.32,64.936667
4,2015-01-01 04:00:00+01:00,56.04,62.906667
5,2015-01-01 05:00:00+01:00,53.63,59.946667
6,2015-01-01 06:00:00+01:00,51.73,56.330000
7,2015-01-01 07:00:00+01:00,51.43,53.800000
8,2015-01-01 08:00:00+01:00,48.98,52.263333
9,2015-01-01 09:00:00+01:00,54.20,50.713333


In [38]:
# no data leakage for first test row
print(f"\nFirst test row (row {split_idx}):")
print(f"  price_actual (target we want to predict) = {df_correct.loc[split_idx, 'price_actual']:.2f}")
print(f"  price_rolling3h (feature given to model)  = {df_correct.loc[split_idx, 'price_rolling3h']:.2f}")

print(f"\nHow was price_rolling3h calculated (with shift)?")
print(f"  Row {split_idx-3}: price = {df.loc[split_idx-3, 'price_actual']:.2f}")
print(f"  Row {split_idx-2}: price = {df.loc[split_idx-2, 'price_actual']:.2f}")
print(f"  Row {split_idx-1}: price = {df.loc[split_idx-1, 'price_actual']:.2f}")

manual_calc_correct = df.loc[split_idx-3:split_idx-1, 'price_actual'].mean()
print(f"\n  Rolling mean = ({df.loc[split_idx-3, 'price_actual']:.2f} + {df.loc[split_idx-2, 'price_actual']:.2f} + {df.loc[split_idx-1, 'price_actual']:.2f}) / 3 = {manual_calc_correct:.2f}")

print(f"\nThe feature only uses PAST values (t-1, t-2, t-3)")


First test row (row 70):
  price_actual (target we want to predict) = 70.10
  price_rolling3h (feature given to model)  = 72.48

How was price_rolling3h calculated (with shift)?
  Row 67: price = 72.14
  Row 68: price = 72.45
  Row 69: price = 72.86

  Rolling mean = (72.14 + 72.45 + 72.86) / 3 = 72.48

The feature only uses PAST values (t-1, t-2, t-3)
